**Industrial LLM Post-Training Experimental Study**

Goal: Adapt a small existing language model to the writing patterns and terminology of our industrial maintenance corpus, using Unsloth + 4-bit QLoRA. This is raw-text next-token prediction, not instruction training.


Process: load preprocessed records → inspect / validation split → load model + tokenizer → inspect tokens → attach LoRA → SFTTrainer with causal LM text + packing → train → record metrics → save adapter → optionally merge. The validation split is a training diagnostic, not our final independent test.

In [1]:
%pip -q install -U unsloth datasets trl peft accelerate

Unsloth optimizes training; Transformers supplies the pretrained architecture and tokenizer; TRL supplies SFTTrainer; datasets reads our JSONL

**Step 1: Import packages and GPU**

We're training a 4-bit model, so a compatible NVIDIA CUDA GPU is required. On Colab a T4 is a reasonable first choice. Import unsloth early, then PyTorch and the trainer utilities. We deliberately do not load SFT or DPO data in this notebook

In [2]:
import unsloth
import os, json, time, random, gc
from pathlib import Path
import torch
import pandas as pd
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

assert torch.cuda.is_available(), "Enable a GPU in Runtime → Change runtime type."
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4
Torch: 2.11.0+cu130


**Step 2: Mount Paths**

In [14]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/llm-post-training-experimental-study')
DATA_FILE = PROJECT_ROOT / 'data/processed/continued_pretraining.jsonl'
ADAPTER_DIR = PROJECT_ROOT / 'models/stage1_non_instruction_adapter'
METRICS_DIR = PROJECT_ROOT / 'results/metrics'
LOCAL_LOGS_DIR = Path('/content/stage1_training_logs')

assert DATA_FILE.exists(), f"Can't find dataset: {DATA_FILE}"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_LOGS_DIR.mkdir(parents=True, exist_ok=True)
print('Training data:', DATA_FILE)
print('Persistent adapter output:', ADAPTER_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training data: /content/drive/MyDrive/llm-post-training-experimental-study/data/processed/continued_pretraining.jsonl
Persistent adapter output: /content/drive/MyDrive/llm-post-training-experimental-study/models/stage1_non_instruction_adapter


**Verification**

In [5]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/llm-post-training-experimental-study"
)

DATA_FILE = PROJECT_ROOT / "data/processed/continued_pretraining.jsonl"

print("Drive mounted:", Path("/content/drive/MyDrive").exists())
print("Project found:", PROJECT_ROOT.exists())
print("Dataset found:", DATA_FILE.exists())

Drive mounted: True
Project found: True
Dataset found: True


**Step 3: Environment Config**

In [6]:
import random
import torch
import os, json, time, random, gc
BASE_MODEL_NAME = "unsloth/Qwen2.5-1.5B-bnb-4bit"
MAX_SEQ_LENGTH = 512
SEED = 42
LORA_R = 16
LORA_ALPHA = 32
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
MAX_STEPS = 30
LEARNING_RATE = 2e-4
WARMUP_STEPS = 5




random.seed(SEED)
torch.manual_seed(SEED)

experiment_config = {
    'base_model': BASE_MODEL_NAME,
    'max_seq_length': MAX_SEQ_LENGTH,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
    'max_steps': MAX_STEPS,
    'learning_rate': LEARNING_RATE,
    'seed': SEED,
    'data_file': str(DATA_FILE),
}
print(json.dumps(experiment_config, indent=2))

{
  "base_model": "unsloth/Qwen2.5-1.5B-bnb-4bit",
  "max_seq_length": 512,
  "lora_r": 16,
  "lora_alpha": 32,
  "batch_size": 1,
  "gradient_accumulation_steps": 8,
  "max_steps": 30,
  "learning_rate": 0.0002,
  "seed": 42,
  "data_file": "/content/drive/MyDrive/llm-post-training-experimental-study/data/processed/continued_pretraining.jsonl"
}


**Step 4: Load Dataset**

In [7]:
from datasets import load_dataset
raw_dataset = load_dataset('json', data_files=str(DATA_FILE), split='train')
assert 'text' in raw_dataset.column_names, raw_dataset.column_names
assert len(raw_dataset) > 0
assert all(isinstance(t, str) and t.strip() for t in raw_dataset['text'])

print('Records:', len(raw_dataset))
print('Columns:', raw_dataset.column_names)
print('Exact duplicate paragraphs:', len(raw_dataset) - len(set(raw_dataset['text'])))
print('First example (preview):', raw_dataset[0]['text'][:450])

Records: 300
Columns: ['text', 'record_id', 'source_type']
Exact duplicate paragraphs: 0
First example (preview): During a routine daytime inspection, a maintenance record for centrifugal pumps described a gradual rise in vibration near the drive-end bearing. The operator noted that the behavior was first noticed during the current shift. From an engineering perspective, the vibration trend may be associated with rotating assembly condition, coupling alignment, or support looseness. For traceability, recorded vibration direction, operating point, and inspect


**Step 5: Train/Validation Split**

In [8]:
splits = raw_dataset.train_test_split(test_size=0.10, seed=SEED)
train_data = splits['train']
validation_data = splits['test']
print('Training records:', len(train_data))
print('Validation records:', len(validation_data))

Training records: 270
Validation records: 30


**Step 6: Load Base Model + Tokenizer**

In [9]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Tokenizer vocabulary:', len(tokenizer))
print('EOS token:', repr(tokenizer.eos_token))
print('PAD token:', repr(tokenizer.pad_token))

==((====))==  Unsloth 2026.11.1: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizer vocabulary: 151665
EOS token: '<|endoftext|>'
PAD token: '<|vision_pad|>'


**Step 7: Inspect Tokensization**

In [10]:
sample_text = train_data[0]['text']
sample_tokens = tokenizer(sample_text, add_special_tokens=True)
print('Characters:', len(sample_text))
print('Tokens:', len(sample_tokens['input_ids']))
print('First 25 IDs:', sample_tokens['input_ids'][:25])
print('First 25 token pieces:', tokenizer.convert_ids_to_tokens(sample_tokens['input_ids'][:25]))

lengths = [len(tokenizer(t, add_special_tokens=False)['input_ids']) for t in raw_dataset['text']]
print('All paragraphs — token count (min/median/max):', min(lengths), sorted(lengths)[len(lengths)//2], max(lengths))
print('Approx total tokens before packing:', sum(lengths))

Characters: 548
Tokens: 83
First 25 IDs: [16014, 264, 2297, 304, 5670, 7479, 11, 264, 13404, 3255, 369, 8628, 505, 331, 14052, 7481, 264, 2297, 304, 5486, 9315, 1283, 13404, 13, 576]
First 25 token pieces: ['During', 'Ġa', 'Ġchange', 'Ġin', 'Ġproduction', 'Ġdemand', ',', 'Ġa', 'Ġmaintenance', 'Ġrecord', 'Ġfor', 'Ġheat', 'Ġex', 'ch', 'angers', 'Ġdescribed', 'Ġa', 'Ġchange', 'Ġin', 'Ġapproach', 'Ġtemperature', 'Ġafter', 'Ġmaintenance', '.', 'ĠThe']
All paragraphs — token count (min/median/max): 77 88 102
Approx total tokens before packing: 26345


In [11]:
prompt = "During a routine inspection, the centrifugal pump"

encoded = tokenizer(prompt, return_tensors="pt")

print("Tokens:", tokenizer.convert_ids_to_tokens(
    encoded["input_ids"][0]
))

print("Decoded:", tokenizer.decode(
    encoded["input_ids"][0]
))

Tokens: ['During', 'Ġa', 'Ġroutine', 'Ġinspection', ',', 'Ġthe', 'Ġcentrif', 'ugal', 'Ġpump']
Decoded: During a routine inspection, the centrifugal pump


**Step 8: LORA setup**

In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
model.print_trainable_parameters()

Unsloth 2026.11.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


**Step 9: Model training config -> next token generation**

In [15]:
training_args = SFTConfig(
    output_dir=str(LOCAL_LOGS_DIR),
    dataset_text_field='text',
    max_length=MAX_SEQ_LENGTH,
    packing=True,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=1,
    save_strategy='no',
    eval_strategy='no',  # manually evaluate once at the end
    report_to='none',
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim='adamw_8bit',
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=validation_data,
    args=training_args,
)
print('Trainer ready. Next: trainer.train()')

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/270 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/270 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/30 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=2):   0%|          | 0/30 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
Trainer ready. Next: trainer.train()


**Step 10: Model Training**

In [16]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start_time = time.perf_counter()
train_result = trainer.train()
torch.cuda.synchronize()

train_seconds = round(time.perf_counter() - start_time, 2)
peak_vram_gb = round(torch.cuda.max_memory_allocated() / (1024**3), 3)

print('Training completed')
print('Time (seconds):', train_seconds)
print('Peak allocated GPU memory (GB):', peak_vram_gb)
print('Final training loss:', train_result.training_loss)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51 | Num Epochs = 5 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss
1,4.091433
2,4.181863
3,4.144452
4,3.957691
5,3.789290
6,3.524036
7,3.250423
8,2.931769
9,2.735202
10,2.598182


Training completed
Time (seconds): 101.32
Peak allocated GPU memory (GB): 2.362
Final training loss: 2.1951297839482624


**Step 11: Model Evaluation**

In [17]:
eval_metrics = trainer.evaluate()
print('Validation metrics:', eval_metrics)

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training Loss,Validation Loss,Step
1.070102,1.231460,30


Validation metrics: {'eval_loss': 1.231460452079773}


In [18]:
import pandas as pd

logs = pd.DataFrame(trainer.state.log_history)

display(
    logs[
        [c for c in ["step", "loss", "eval_loss"]
         if c in logs.columns]
    ].dropna(how="all")
)

,step,loss,eval_loss
0,1,4.091433,NaN
1,2,4.181863,NaN
2,3,4.144452,NaN
3,4,3.957691,NaN
4,5,3.789290,NaN
5,6,3.524036,NaN
6,7,3.250423,NaN
7,8,2.931769,NaN
8,9,2.735202,NaN
9,10,2.598182,NaN


**Step 12: Save adapter and tokenizer in drive**

In [19]:
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

metrics = {
    **experiment_config,
    'train_records': len(train_data),
    'validation_records': len(validation_data),
    'training_loss': float(train_result.training_loss),
    'validation_loss': float(eval_metrics['eval_loss']) if 'eval_loss' in eval_metrics else None,
    'training_seconds': train_seconds,
    'peak_gpu_memory_allocated_gb': peak_vram_gb,
    'gpu_name': torch.cuda.get_device_name(0),
    'adapter_dir': str(ADAPTER_DIR),
}
metric_file = METRICS_DIR / 'stage1_non_instruction_metrics.json'
metric_file.write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding='utf-8')

assert (ADAPTER_DIR / 'adapter_config.json').exists(), 'Adapter save incomplete'
assert metric_file.exists()
print('Adapter saved:', ADAPTER_DIR)
print('Metrics saved:', metric_file)
print('Adapter files:', [p.name for p in ADAPTER_DIR.iterdir()])

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/llm-post-training-experimental-study/models/stage1_non_instruction_adapter/tokenizer_config.json.


Adapter saved: /content/drive/MyDrive/llm-post-training-experimental-study/models/stage1_non_instruction_adapter
Metrics saved: /content/drive/MyDrive/llm-post-training-experimental-study/results/metrics/stage1_non_instruction_metrics.json
Adapter files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'tokenizer.json', 'tokenizer.model']


**Inference/Testing**

In [20]:
import torch
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

prompt = (
    "During a routine inspection of a centrifugal pump, "
    "the maintenance team observed"
)
prompt =(
"During a routine daytime inspection, a maintenance record for centrifugal pumps described a gradual rise in vibration"
)
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.15,
        no_repeat_ngram_size=4,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode only the NEW tokens generated
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

print(
    tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )
)

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 over the previous shift. The equipment had recently returned to service following a scheduled downtime event. From an engineering perspective, friction or misalignment may be contributing; for traceability, measurement consistency and sampling interval should be noted. For traceability, trend duration and operating conditions should be documented together with identified trends; whether replacement is required depends on the asset’s acceptance criteria. From an operator perspective, trending behavior relative to loading and instrumentation uncertainty should be considered; post-maintenance comparisons without context are not useful.


In [27]:
import torch
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
model.eval()

prompt = (
    "During a routine daytime inspection, "
    "a maintenance record for centrifugal pumps"
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]

print(tokenizer.decode(generated, skip_special_tokens=True))

Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


, a centrifugal pumps, a centrifugal pumps, a centrifugal pumps, a centrifugal pumps, a centrifugal pumps, a centrifugal pumps, a centrifugal pumps, a centrif


In [3]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-bnb-4bit",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True
)

FastLanguageModel.for_inference(model)
model.eval()

prompt = (
    "Maintenance inspection report:\n"
    "Equipment: Centrifugal pump P-204\n"
    "Observation: Vibration increased near the drive-end bearing.\n"
    "Additional findings:"
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]

print(tokenizer.decode(generated, skip_special_tokens=True))

==((====))==  Unsloth 2026.11.1: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Bearing wear has occurred at the drive-end.
Solution: Reinstall bearings, perform maintenance and inspection on equipment.
Based on the above information, which of the following options is correct?
A. The vibration caused by the increased bearing wear is related to the centrifugal pump P-204.
B. The vibration is unrelated to the increased bearing wear in this case.
Answer:

A

In the


In [1]:
import torch
from unsloth import FastLanguageModel

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/tinyllama-bnb-4bit",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(base_model)
base_model.eval()

prompt = "The capital of France is"

inputs = base_tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.inference_mode():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=base_tokenizer.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

print(base_tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.1: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Unsloth: Will load unsloth/tinyllama-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


the main government of France is the main government of the main government of France is the main government of France is the main government of France is the main government of France is the main government of France is


In [20]:
#Optional Merge
DO_MERGE = False
if DO_MERGE:
    MERGED_DIR = PROJECT_ROOT / 'models/stage1_non_instruction_merged_16bit'
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    FastLanguageModel.for_training(model)
    model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method='merged_16bit')
    print('Saved merged model:', MERGED_DIR)
else:
    print('Skipped merge — saved adapter is sufficient for this stage.')

Skipped merge — saved adapter is sufficient for this stage.


TinyLama gave incorrect results --> repeated text on inference
So we switched to Qwen2.5-1.5B

## Observations and Experimental Findings

### 1. Initial Experiment — TinyLlama 1.1B

- Continued pretraining was initially performed on TinyLlama 1.1B using QLoRA and a synthetic industrial maintenance dataset containing 300 paragraphs.
- The model was trained for 30 optimization steps, obtaining a final recorded training loss of approximately 4.65 and validation loss of 4.75.
- During inference, the fine-tuned model exhibited repetitive text generation, incoherent sentence construction, and occasional invented terminology.
- Similar repetitive behavior was observed when testing the original pretrained TinyLlama model, including on general prompts unrelated to industrial maintenance.
- This suggested that the observed generation problems could not be attributed exclusively to fine-tuning or the industrial dataset.
- The available loss measurements were insufficient to establish whether the model had overfitted.

### 2. Model Selection — Transition to Qwen2.5-1.5B

- Following the initial TinyLlama experiment, Qwen2.5-1.5B Base was selected for further investigation.
- Initial inference using the pretrained Qwen model produced more coherent technical text than TinyLlama in the prompts tested.
- However, some responses drifted into unrelated multiple-choice question formats or presented unsupported technical diagnoses.
- These observations highlighted the limitations of evaluating pretrained language models using isolated text-generation examples.

### 3. Domain-Adapted Qwen — Post-Fine-Tuning Inference

Following non-instruction fine-tuning, the Qwen model generated a coherent continuation of an industrial maintenance observation involving increasing centrifugal-pump vibration.

The generated response demonstrated:

- **Domain-relevant vocabulary:** Appropriate references to vibration trends, alignment, loading conditions, instrumentation uncertainty, and maintenance documentation.
- **Contextual continuity:** The generated text largely maintained the industrial maintenance context of the input prompt.
- **Improved observed coherence:** The response was substantially more readable than the repetitive TinyLlama outputs observed earlier.
- **Uncertainty awareness:** Potential contributing factors were expressed as possibilities rather than definitive fault diagnoses.
- **Minor stylistic repetition:** Phrases such as "For traceability" appeared more than once.
- **Unsupported contextual additions:** The model introduced information about recent equipment downtime that was not supplied in the prompt.

These findings indicate that the domain-adapted Qwen model was capable of generating plausible industrial maintenance prose in the tested example. However, the observations do not independently establish that fine-tuning improved performance relative to the original Qwen checkpoint.

### 4. Limitations

- The training corpus consisted of only 300 synthetically generated paragraphs, limiting its diversity and representativeness.
- Repeated linguistic structures in the dataset may influence the model's generated writing style.
- Generation quality was assessed through a small number of qualitative examples rather than a systematic evaluation benchmark.
- A controlled comparison of base-model and fine-tuned validation losses has not yet been documented.
- Coherent text generation does not necessarily imply factual accuracy, reliable diagnosis, or domain expertise.

### 5. Key Takeaways

1. Non-instruction fine-tuning adapts a model through causal language modeling rather than explicitly teaching it to follow user instructions.
2. Model selection and baseline evaluation are important before attributing generation problems to fine-tuning.
3. Training loss and validation loss alone are insufficient to establish successful domain adaptation.
4. Generated text should be evaluated for coherence, contextual relevance, unsupported claims, and repetitive patterns.
5. Continued pretraining does not automatically guarantee better factual accuracy or instruction-following behavior.

### 6. Next Steps

The next stage will involve supervised instruction fine-tuning (SFT), using structured instruction-response examples to train the model to transform unstructured maintenance observations into standardized reports.

Subsequent experiments will investigate preference optimization using DPO, parameter-efficient training configurations, and comparative evaluation across model checkpoints.

**Conclusion:** This preliminary experiment demonstrated the implementation of a QLoRA-based domain-adaptation pipeline and produced a coherent industrial maintenance text continuation using the fine-tuned Qwen model. Further controlled evaluation is necessary to determine whether continued pretraining provides measurable improvements over the pretrained baseline.